# 02 · RQ1: emotion recognition (H1a, H1b, H3a)

Analysis plan §3. Task A outputs of both models against the adjudicated gold labels, with merges applied to gold and predictions alike. Invalid outputs count as incorrect.

In [ ]:
# Shared setup: every path comes from payo_eval.paths, so PAYO_* variables can point the notebook at synthetic data
import warnings

import matplotlib.pyplot as plt
import pandas as pd

from payo_eval import analyses as A
from payo_eval import load
from payo_eval.paths import get_paths, load_config

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

In [ ]:
# Settings from 03-evaluation/config.toml; outputs go to results/02-emotion-recognition/ and figures/
PATHS = get_paths()
CFG = load_config(PATHS)
SEED = CFG["seed"]
B = CFG["analysis"]["bootstrap_b"]
RESULTS = PATHS.results / "02-emotion-recognition"
TESTS = PATHS.results / "tests"
FIGURES = PATHS.figures
for folder in (RESULTS, TESTS, FIGURES):
    folder.mkdir(parents=True, exist_ok=True)


def save(table: pd.DataFrame, stem: str) -> pd.DataFrame:
    """Write a result table as CSV and return it for display."""
    table.to_csv(RESULTS / f"{stem}.csv", index=False)
    return table

In [ ]:
# Task A outputs joined with gold (one row per post x model)
pred = load.predictions(PATHS)
print(len(pred), "rows;", pred["post_id"].nunique(), "posts")

## Accuracy and macro-F1 with post-bootstrap CIs

Overall, by `emotion_type` and by `expression`.

In [ ]:
save(A.recognition_metrics(pred, B, SEED), "metrics")

In [ ]:
# Per-class metrics, confusion matrices, majority baseline and the concept metrics (alias vs. strict, v1.1)
tables = A.recognition_tables(pred)
for stem, table in tables.items():
    table.to_csv(RESULTS / f"{stem}.csv")
tables["baseline"]

In [ ]:
tables["concept"]

In [ ]:
# Confusion matrices as heatmaps (rows = gold, columns = prediction)
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, model in zip(axes, ("gemma", "sealion")):
    table = tables[f"confusion_{model}"]
    ax.imshow(table.to_numpy(), cmap="Blues")
    ax.set_xticks(range(table.shape[1]), table.columns, rotation=90)
    ax.set_yticks(range(table.shape[0]), table.index)
    ax.set_title(model)
fig.tight_layout()
fig.savefig(FIGURES / "02_confusion.png", dpi=150)

## Tests: GEE (H1a, H1b, H3a) and McNemar

Odds ratios from the logistic GEE clustered by post, on culture-bound and shared posts only (`none_practical` excluded, §1). Directional p-values; Holm in notebook 07.

In [ ]:
tests, mcnemar = A.recognition_tests(pred)
tests.to_csv(TESTS / "F1_recognition.csv", index=False)
tests

In [ ]:
save(mcnemar, "mcnemar")

## Calibration (descriptive only, v1.1)

The pilot showed near-constant confidence, so this is not interpreted as a calibration finding.

In [ ]:
summary, bins = A.calibration(pred)
bins.to_csv(RESULTS / "calibration_bins.csv", index=False)
save(summary, "calibration")

## Findings

_Fill in after the real run: the main estimates with their CIs, and anything flagged above._